# Demo program for langGraph for Single-Agent

In [2]:
from langgraph.graph import StateGraph
import os
from langchain_groq import ChatGroq

def ask_question(state):
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )
    response = llm.invoke("Give me one motivational quote.")
    return {"output": response.content}

graph = StateGraph(dict)
graph.add_node("ask", ask_question)
graph.set_entry_point("ask")
graph.set_finish_point("ask")

app = graph.compile()
result = app.invoke({})
print(result["output"])

“Believe in yourself, push your limits, and never give up—because the only thing standing between you and your dreams is the story you keep telling yourself.”


# Demo program for langchain for Single-Agent

In [5]:
from langchain_core.prompts import PromptTemplate 
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
prompt = PromptTemplate.from_template("What is a fun fact about {topic}?")
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "cats"})
print(result)

Sure! Did you know that cats have a “righting reflex” that allows them to twist their bodies in mid‑air and land on their feet? Even a kitten as young as three weeks old can start to right itself, and by the time they’re six weeks old they can usually land upright from a fall. This amazing ability is thanks to their flexible spine, lack of a collarbone, and a finely tuned sense of balance. It’s why you’ll often see cats making spectacular leaps and seemingly defying gravity! 🐱✨


# Demo program for langGraph for Multi-Agent

In [7]:
import os
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the LLM
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

# 2. Define the Researcher Agent
researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert tech researcher. Provide 3 key factual bullet points about the topic requested."),
    ("user", "Research the following topic: {topic}")
])
researcher_agent = researcher_prompt | llm | StrOutputParser()

# 3. Define the Writer Agent 
writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite tech blogger. Take raw research data and expand it into a short, engaging LinkedIn post."),
    ("user", "Here is the raw research: \n\n{research_data}")
])
writer_agent = writer_prompt | llm | StrOutputParser()

# 4. Orchestrate the execution sequentially (Multi-Agent Flow)
def run_langchain_multi_agent(topic: str):
    print(f"--- [Researcher] Fetching facts for: '{topic}' ---")
    research_notes = researcher_agent.invoke({"topic": topic})
    print(f"Research Complete:\n{research_notes}\n")
    
    print("--- [Writer] Generating LinkedIn Post ---")
    final_post = writer_agent.invoke({"research_data": research_notes})
    return final_post

# Execute Demo
if __name__ == "__main__":
    topic_input = "LangGraph for multi-agent workflows"
    result = run_langchain_multi_agent(topic_input)
    print(f"Final Output:\n{result}")


--- [Researcher] Fetching facts for: 'LangGraph for multi-agent workflows' ---
Research Complete:
- **LangGraph is an open‑source Python library that extends LangChain to enable graph‑structured, multi‑agent workflows**, allowing developers to define nodes (agents, tools, or functions) and explicit edges that dictate the flow of information and control between them.

- **It provides built‑in state management and loop detection**, so multi‑agent pipelines can maintain context across turns, handle dynamic branching (e.g., conditional routing, parallel execution), and automatically prevent infinite recursion when agents invoke each other.

- **LangGraph integrates with LangChain’s memory, tool‑calling, and retrieval utilities**, making it straightforward to build complex orchestrations such as “planner → specialist agents → validator → aggregator” or hierarchical task decomposition, while supporting streaming responses and easy debugging via visual graph visualizations.

--- [Writer] Gene

# Demo program for langchain for Multi-Agent

In [8]:
import os
from langchain_groq import ChatGroq
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

# 1. Define the Shared State Schema
class AgentState(TypedDict):
    topic: str
    research_notes: str
    final_draft: str

# 2. Initialize LLM
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

# 3. Define the Node Functions (The Agents)
def researcher_node(state: AgentState):
    print("--- [LangGraph Node: Researcher] Gathering Facts ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a tech researcher. Return 3 crucial facts about the topic."),
        ("user", "{topic}")
    ])
    chain = prompt | llm
    response = chain.invoke({"topic": state["topic"]})
    
    # Return the dictionary key to update the global shared state
    return {"research_notes": response.content}

def writer_node(state: AgentState):
    print("--- [LangGraph Node: Writer] Writing Post ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a professional blogger. Write a LinkedIn post using these notes."),
        ("user", "{research_notes}")
    ])
    chain = prompt | llm
    response = chain.invoke({"research_notes": state["research_notes"]})
    
    # Update the final draft key in the global state
    return {"final_draft": response.content}

# 4. Build and Compile the Graph Machine
builder = StateGraph(AgentState)

# Add agents as graph nodes
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)

# Construct flow pathways (Edges)
builder.add_edge(START, "researcher")  # Entry point
builder.add_edge("researcher", "writer")  # Hand-off from Researcher to Writer
builder.add_edge("writer", END)         # Exit point

# Compile into an executable application
graph_app = builder.compile()

# Execute Demo
if __name__ == "__main__":
    initial_state = {"topic": "LangGraph for multi-agent workflows", "research_notes": "", "final_draft": ""}
    output_state = graph_app.invoke(initial_state)
    
    print(f"\nFinal State Output:\n{output_state['final_draft']}")


--- [LangGraph Node: Researcher] Gathering Facts ---
--- [LangGraph Node: Writer] Writing Post ---

Final State Output:
🚀 **LangGraph & Multi‑Agent Workflows – 3 Things Every AI Engineer Should Know** 🚀  

If you’ve been experimenting with LangChain, you’ve probably felt the friction of wiring dozens of agents, tools, and custom functions together. LangGraph flips the script – it turns that tangled spaghetti into a clean, **graph‑based orchestration** that scales with confidence. Here are the three game‑changing facts that make it a must‑try:

---

### 1️⃣ Composable Graph‑Based Orchestration  
🔹 **Nodes = LangChain components** – LLM chains, tools, agents, or any custom function.  
🔹 **Edges = data & control flow** – route messages dynamically based on runtime signals (tool success, confidence thresholds, user intent, etc.).  
🔹 **Plug‑and‑play** – swap a specialist agent in or out without touching the rest of the pipeline.  

*Result:* Build, test, and iterate multi‑agent systems the